# Análisis de errores

Usa `results/evaluation/test_predictions.csv` generado en
`final_evaluation.ipynb` — no necesita recargar ningún modelo.

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../results/evaluation/test_predictions.csv')
df.head()

,id,text,is_humor,votes_no,votes_1,votes_2,votes_3,votes_4,votes_5,funniness_average,pred_nbsvm,prob_nbsvm,pred_baseline,prob_baseline,pred_lora,prob_lora
0,441341394196520960,"#3PalabrasSumamenteDolorosas ""Ingrese contrase...",1,2,4,4,0,0,0,1.5,1,0.671694,0,0.137054,0,0.142594
1,944735834711969792,"Ir caminando por la calle, que te saluden desd...",0,3,2,0,0,0,0,NaN,1,0.859408,0,0.069118,0,0.475027
2,518425086664396800,"Pa, en el ejército me violó un negro! -La vida...",1,0,3,2,0,0,0,1.4,1,0.911918,1,0.993269,1,0.999949
3,965760641934950400,tengo hambre sabe?,0,3,0,0,0,0,0,NaN,0,0.137522,0,0.001916,0,0.000035
4,942039418394816512,"Cuanto más tiempo esperas algo, más lo aprecia...",0,8,0,0,1,0,0,NaN,0,0.147918,0,0.029382,0,0.009166


In [3]:
label_col = 'is_humor'  # ajustar si la columna se llama distinto en el CSV
text_col  = 'text'

df['error_nbsvm']    = df[label_col] != df['pred_nbsvm']
df['error_baseline'] = df[label_col] != df['pred_baseline']
df['error_lora']     = df[label_col] != df['pred_lora']

## 1. Tasa de error por clase
% de falsos negativos sobre el total de humor real, % de falsos positivos
sobre el total de no-humor real — para cada uno de los 3 modelos.

In [4]:
def error_rate_by_class(df, pred_col, label_col):
    humor_real = df[df[label_col] == 1]
    no_humor_real = df[df[label_col] == 0]

    fn_rate = (humor_real[pred_col] == 0).mean()       # humor real predicho como no-humor
    fp_rate = (no_humor_real[pred_col] == 1).mean()    # no-humor real predicho como humor
    return fn_rate, fp_rate

rows = []
for modelo, pred_col in [('NBSVM', 'pred_nbsvm'), ('BETO baseline', 'pred_baseline'), ('BETO + LoRA', 'pred_lora')]:
    fn_rate, fp_rate = error_rate_by_class(df, pred_col, label_col)
    rows.append({'modelo': modelo, 'tasa_falsos_negativos': fn_rate, 'tasa_falsos_positivos': fp_rate})

df_error_rates = pd.DataFrame(rows)

os.makedirs('results/evaluation', exist_ok=True)
df_error_rates.to_csv('results/evaluation/error_rates_by_class.csv', index=False)
df_error_rates

,modelo,tasa_falsos_negativos,tasa_falsos_positivos
0,NBSVM,0.398488,0.106513
1,BETO baseline,0.211663,0.107870
2,BETO + LoRA,0.163067,0.127544


## 2. Falsos positivos y negativos más confiados (3 modelos)
Los que el modelo predijo con mayor seguridad pero estaban mal.

In [5]:
def top_confident_errors(df, pred_col, prob_col, label_col, text_col, n=5):
    errores = df[df[label_col] != df[pred_col]].copy()

    fp = errores[errores[pred_col] == 1].nlargest(n, prob_col)
    fn = errores[errores[pred_col] == 0].nsmallest(n, prob_col)

    return fp[[text_col, label_col, pred_col, prob_col]], fn[[text_col, label_col, pred_col, prob_col]]

In [6]:
print('=== NBSVM ===')
fp_nbsvm, fn_nbsvm = top_confident_errors(df, 'pred_nbsvm', 'prob_nbsvm', label_col, text_col)
print('\nFalsos positivos más confiados:')
display(fp_nbsvm)
print('\nFalsos negativos más confiados:')
display(fn_nbsvm)

=== NBSVM ===

Falsos positivos más confiados:


,text,is_humor,pred_nbsvm,prob_nbsvm
1418,"""Cormillot , Fabbiani es argentino"" #Chistes",0,1,0.997278
1865,Terminando el examen de Matemáticas: -Están li...,0,1,0.972957
689,Macri le dijo las gordas que usar calzas está ...,0,1,0.953683
1903,"Llega un tipo y pregunta: -Señor, ¿tiene cebol...",0,1,0.952995
2173,"—Hijo, ¿Agarraste mi disco de Pedrito Fernánde...",0,1,0.936054



Falsos negativos más confiados:


,text,is_humor,pred_nbsvm,prob_nbsvm
271,-¿Hacemos una vaquita pa los bizcochos? -Calma...,1,0,0.045278
1047,No se sentía tanta tristeza en México desde qu...,1,0,0.050457
111,"«Wey, la neta no es por ser culero pero» —Algu...",1,0,0.056042
32,HALA... Eropuerto España JAJAJAJAJ 😂😂😂😂😂,1,0,0.059776
2045,No contesto Dm pero puedes hablarme por Paypal.,1,0,0.062722


In [7]:
print('=== BETO baseline ===')
fp_baseline, fn_baseline = top_confident_errors(df, 'pred_baseline', 'prob_baseline', label_col, text_col)
print('\nFalsos positivos más confiados:')
display(fp_baseline)
print('\nFalsos negativos más confiados:')
display(fn_baseline)

=== BETO baseline ===

Falsos positivos más confiados:


,text,is_humor,pred_baseline,prob_baseline
2173,"—Hijo, ¿Agarraste mi disco de Pedrito Fernánde...",0,1,0.998964
148,El HOMBRE es un ser tan dependiente que hasta ...,0,1,0.997006
455,- “¿tú eres hetero?” - no... - la gente hetero...,0,1,0.994215
1865,Terminando el examen de Matemáticas: -Están li...,0,1,0.992984
764,El Seba acaba de hacer un GOLAZO y anoche carr...,0,1,0.992721



Falsos negativos más confiados:


,text,is_humor,pred_baseline,prob_baseline
260,Vivir tu vida es algo tan difícil que nadie an...,1,0,0.005720
508,No dejes para mañana lo que puedes hacer pasad...,1,0,0.013374
297,Cuando muere un mimo hago un minuto de ruido,1,0,0.022113
132,Mamado Nervo Jose Vasconpelos Gabriel García M...,1,0,0.026938
1315,"Eres como mi tarea, nunca me acuerdo de ti.",1,0,0.031052


In [8]:
print('=== BETO + LoRA ===')
fp_lora, fn_lora = top_confident_errors(df, 'pred_lora', 'prob_lora', label_col, text_col)
print('\nFalsos positivos más confiados:')
display(fp_lora)
print('\nFalsos negativos más confiados:')
display(fn_lora)

=== BETO + LoRA ===

Falsos positivos más confiados:


,text,is_humor,pred_lora,prob_lora
1085,"RT : Cocodrilo: ""¿Voy a comerme a tu niño? Res...",0,1,0.999881
455,- “¿tú eres hetero?” - no... - la gente hetero...,0,1,0.999632
1903,"Llega un tipo y pregunta: -Señor, ¿tiene cebol...",0,1,0.999632
1595,"Mientras tu me ignoras, la señora de las pelíc...",0,1,0.999582
2173,"—Hijo, ¿Agarraste mi disco de Pedrito Fernánde...",0,1,0.999550



Falsos negativos más confiados:


,text,is_humor,pred_lora,prob_lora
413,Hoy le tocó a nsync estar al mando de mi cerebro,1,0,0.000247
215,"No has vivido, si nunca dijiste ""Mama si me co...",1,0,0.000533
891,"Me estoy poniendo en forma, pero todavía no se...",1,0,0.000619
1653,"Odio cuando tengo rabia y me hacen reír, me ar...",1,0,0.000669
2119,Tuve una quesadilla,1,0,0.000825


## 3. Errores comunes a los 3 modelos
Tweets donde NBSVM, baseline y LoRA fallan los tres simultáneamente —
posibles "casos imposibles" (sarcasmo extremo, ambigüedad genuina).

In [9]:
errores_comunes = df[df['error_nbsvm'] & df['error_baseline'] & df['error_lora']]
print(f'Tweets donde fallan los 3 modelos: {len(errores_comunes)} de {len(df)} ({100*len(errores_comunes)/len(df):.1f}%)')

errores_comunes[[text_col, label_col, 'pred_nbsvm', 'pred_baseline', 'pred_lora']].to_csv(
    'results/evaluation/errores_comunes_3_modelos.csv', index=False
)
errores_comunes[[text_col, label_col, 'pred_nbsvm', 'pred_baseline', 'pred_lora']]

Tweets donde fallan los 3 modelos: 139 de 2400 (5.8%)


,text,is_humor,pred_nbsvm,pred_baseline,pred_lora
32,HALA... Eropuerto España JAJAJAJAJ 😂😂😂😂😂,1,0,0,0
51,No me dejan poner los 4 porteros que he llevad...,1,0,0,0
54,Al árbitro del Milan-Barça le gustó como le al...,0,1,1,1
55,Me encantan las publicidades que te dicen con ...,1,0,0,0
60,"No manche jefa, claro que es normal que para e...",1,0,0,0
...,...,...,...,...,...
2294,Nubes bajas Acarician tus lanas En un insomnio...,0,1,1,1
2359,"Yo no guardo rencor a mis ex's, les guardo flo...",1,0,0,0
2362,Extrañar es mejor que estar cerca y que esté r...,1,0,0,0
2370,A Ramos le entran las críticas por un oído y l...,1,0,0,0


## 4. Comparación cruzada NBSVM vs BETO vs LoRA
Dónde un modelo acierta y el otro falla.

In [10]:
def comparacion_cruzada(df, error_col_a, error_col_b, nombre_a, nombre_b):
    solo_a_falla = df[df[error_col_a] & ~df[error_col_b]]
    solo_b_falla = df[~df[error_col_a] & df[error_col_b]]
    print(f'{nombre_a} falla y {nombre_b} acierta: {len(solo_a_falla)}')
    print(f'{nombre_b} falla y {nombre_a} acierta: {len(solo_b_falla)}')
    return solo_a_falla, solo_b_falla

In [11]:
print('=== NBSVM vs BETO + LoRA ===')
nbsvm_falla, lora_falla = comparacion_cruzada(df, 'error_nbsvm', 'error_lora', 'NBSVM', 'LoRA')

print('\nEjemplos donde NBSVM falla y LoRA acierta (LoRA capta algo que NBSVM no):')
display(nbsvm_falla[[text_col, label_col, 'pred_nbsvm', 'pred_lora']].head(10))

print('\nEjemplos donde LoRA falla y NBSVM acierta:')
display(lora_falla[[text_col, label_col, 'pred_nbsvm', 'pred_lora']].head(10))

=== NBSVM vs BETO + LoRA ===
NBSVM falla y LoRA acierta: 342
LoRA falla y NBSVM acierta: 155

Ejemplos donde NBSVM falla y LoRA acierta (LoRA capta algo que NBSVM no):


,text,is_humor,pred_nbsvm,pred_lora
1,"Ir caminando por la calle, que te saluden desd...",0,1,0
5,Si se la dan a #Canelo es un pinche robo y de ...,0,1,0
30,-Te amo. -Yo tampoco.,1,0,1
34,"Entrar al banco y gritar: ""¡BOMBAAA...para bai...",1,0,1
35,Quisiera ser pobre un sólo día. Porque serlo t...,1,0,1
36,"Yo de policía le habría gritado ""auxilio una b...",0,1,0
39,Ya sólo faltan las películas de “Mi pobre ange...,1,0,1
41,Como se llama la canción que sale en el bar de...,0,1,0
46,"""No sé que están diciendo estos idiotas pero m...",1,0,1
48,-Me compré una camioneta con los asientos de l...,1,0,1



Ejemplos donde LoRA falla y NBSVM acierta:


,text,is_humor,pred_nbsvm,pred_lora
0,"#3PalabrasSumamenteDolorosas ""Ingrese contrase...",1,1,0
16,Soy el obeso negro se la familia.,1,1,0
28,"-Pará, te enojas por todo. -Debe ser porque la...",0,0,1
98,Camiseta vieja = ¡Pijama Nuevo!.,0,0,1
101,El ex líder de está banda como venía anunciand...,0,0,1
104,Te quiero tanto que si te mando a la mierda te...,0,0,1
110,Si alguien me da una Sori o una Hani no me quejo.,0,0,1
113,"-Andá a dormir jejeje. -¡La puta madre, vos de...",0,0,1
114,"No se si estoy cruda, peda, viva, sola, arriba...",0,0,1
135,Los argentinos como siempre chiflando el himno...,0,0,1


In [12]:
print('=== BETO baseline vs BETO + LoRA ===')
baseline_falla, lora_falla_2 = comparacion_cruzada(df, 'error_baseline', 'error_lora', 'BETO baseline', 'LoRA')

print('\nEjemplos donde baseline falla y LoRA acierta (el fine-tuning eficiente ayuda):')
display(baseline_falla[[text_col, label_col, 'pred_baseline', 'pred_lora']].head(10))

=== BETO baseline vs BETO + LoRA ===
BETO baseline falla y LoRA acierta: 130
LoRA falla y BETO baseline acierta: 114

Ejemplos donde baseline falla y LoRA acierta (el fine-tuning eficiente ayuda):


,text,is_humor,pred_baseline,pred_lora
15,Ultimo dia de clases #ingles #vacaciones 10 dí...,0,1,0
26,"#YoSiMeAtrevo a decirle que soy Bipolar, ahora...",1,0,1
30,-Te amo. -Yo tampoco.,1,0,1
33,Desconfío de la gente que usa muchos kg en eje...,0,1,0
36,"Yo de policía le habría gritado ""auxilio una b...",0,1,0
52,EN QUE SE PARECE BARCELONA A RIVER ??? QUE BAR...,1,0,1
57,Este año el 2012 MTV Video Music Awards está d...,0,1,0
91,"Facebook necesita un botón que diga: No mames,...",0,1,0
96,"¿Alguien había notado que ""OK"" es como un dibu...",0,1,0
141,"#carmengloriambd Muy buenos dias, la pension d...",0,1,0
